# Multi-view Pig Posture Recognition — lokalny full-auto all-fold training

Notebook do uruchomienia **lokalnie w Jupyter / VS Code / PyCharm**, zakładający że plik `kaggle.json` leży w tym samym folderze co notebook.

Pipeline:

1. konfiguruje lokalne ścieżki względem katalogu notebooka,
2. kopiuje `kaggle.json` do `~/.kaggle/kaggle.json`, jeśli trzeba,
3. pobiera i rozpakowuje dane konkursowe z Kaggle,
4. standaryzuje dataframe'y,
5. tworzy foldy bez przecieku domenowego,
6. trenuje automatycznie wszystkie foldy dla modelu ustawionego w komórce `CFG`,
7. zapisuje checkpointy w sposób odporny na problem `torch.load` / `weights_only`,
8. robi inference na test jako ensemble checkpointów ze wszystkich foldów,
9. zapisuje finalny `submission_ensemble_all_folds.csv`.

Najważniejsze: **model i hiperparametry zmieniasz tylko w komórce `CFG`**.


## 0. Instalacja zależności

Uruchom tę komórkę w swoim lokalnym środowisku Pythona. Jeżeli masz już zainstalowany PyTorch z odpowiednim CUDA, ta komórka go nie nadpisuje bezpośrednio — instaluje głównie biblioteki pomocnicze.

Jeśli `torch.cuda.is_available()` później zwróci `False`, problem leży zwykle w lokalnej instalacji PyTorch/CUDA, nie w notebooku.


In [ ]:
%pip install -q timm albumentations kaggle torchmetrics opencv-python-headless iterative-stratification

import os
import re
import gc
import glob
import json
import math
import random
import shutil
import zipfile
import warnings
import subprocess
import sys
from pathlib import Path
from collections import defaultdict, Counter

import cv2
import timm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix

try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
except Exception as e:
    raise RuntimeError("Albumentations import failed. Zrestartuj kernel po instalacji i uruchom notebook ponownie.") from e

warnings.filterwarnings("ignore")
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 1. Konfiguracja eksperymentu

Najważniejsze przełączniki:

- `MODEL_NAME` — backbone z `timm`.
- `IMG_SIZE` — sugeruję zacząć od 384.
- `CROP_PADDING` — kontekst wokół bboxa; często 0.15–0.30 działa lepiej niż tight crop.
- `USE_CAMERA_EMBEDDING` — dodaje embedding kamery do klasyfikatora.
- `CV_MODE` — domyślnie `camera_group`, czyli walidacja bardziej odporna na domain shift.
- `USE_WEIGHTED_SAMPLER` — ostrożnie; może pomóc przy niezbalansowanych klasach.

In [ ]:
class CFG:
    # Reproducibility
    SEED = 42

    # Paths — lokalnie wszystko jest względne względem katalogu, z którego działa notebook.
    # Założenie: kaggle.json leży w tym samym folderze co notebook.
    COMPETITION = "multi-view-pig-posture-recognition"
    ROOT = Path.cwd()
    DATA_DIR = ROOT / "data"
    WORK_DIR = ROOT / "pig_posture_work"
    OUT_DIR = WORK_DIR / "outputs"
    REPO_DIR = ROOT / "PigPostureComputerVision"
    KAGGLE_JSON = ROOT / "kaggle.json"

    # Model
    # Dobre pierwsze próby:
    # "tf_efficientnetv2_s.in21k_ft_in1k"
    # "convnext_tiny.fb_in22k_ft_in1k"
    # "swin_tiny_patch4_window7_224.ms_in22k_ft_in1k"
    MODEL_NAME = "convnext_tiny.fb_in22k_ft_in1k"
    IMG_SIZE = 384
    NUM_CLASSES = 5

    # Klasy zgodne z wcześniejszymi notebookami.
    # Jeśli CSV ma string labels, mapowanie zostanie dopasowane automatycznie niżej.
    CLASS_NAMES = [
        "Lateral_lying_left",
        "Lateral_lying_right",
        "Sitting",
        "Standing",
        "Sternal_lying",
    ]

    # Label-aware horizontal flip: 0 <-> 1
    FLIP_LABEL_MAP = {0: 1, 1: 0, 2: 2, 3: 3, 4: 4}

    # Data / CV
    N_FOLDS = 5
    FOLD = 0
    CV_MODE = "camera_group"  # "camera_group", "image_group", "stratified"
    CROP_PADDING = 0.20
    MIN_GROUPS_FOR_GROUPKOLD = 5

    # Training
    EPOCHS = 8
    BATCH_SIZE = 16
    NUM_WORKERS = 2  # na Windowsie, jeśli DataLoader się wiesza, ustaw 0
    LR = 2e-4
    WEIGHT_DECAY = 1e-4
    WARMUP_EPOCHS = 1
    GRAD_CLIP = 1.0
    USE_AMP = True
    USE_WEIGHTED_SAMPLER = False
    LABEL_SMOOTHING = 0.05

    # Camera-aware head
    USE_CAMERA_EMBEDDING = True
    CAMERA_EMB_DIM = 16

    # TTA
    USE_TTA = True
    TTA_HFLIP = True  # poprawnie remapuje klasy 0/1
    TTA_FIVE_CROP = False  # droższe; zwykle dopiero finalnie

    # Pseudo-labeling, domyślnie wyłączone.
    USE_PSEUDO_LABELS = False
    PSEUDO_CONF_THRESHOLD = 0.97
    PSEUDO_LOSS_WEIGHT = 0.3

    # Multi-view / same-event aggregation.
    # Jeśli nazwy plików pozwalają wykryć ten sam moment z różnych kamer, można uśrednić logity.
    USE_GROUP_AGGREGATION = True

CFG.OUT_DIR.mkdir(parents=True, exist_ok=True)
CFG.WORK_DIR.mkdir(parents=True, exist_ok=True)
CFG.DATA_DIR.mkdir(parents=True, exist_ok=True)


def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_everything(CFG.SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("ROOT:", CFG.ROOT)
print("DATA_DIR:", CFG.DATA_DIR)
print("OUT_DIR:", CFG.OUT_DIR)
print("device:", device)


## 2. Pobranie danych z Kaggle i opcjonalne sklonowanie repo

Ten notebook zakłada, że `kaggle.json` leży w tym samym folderze co notebook.

Komórka:

1. sprawdza `./kaggle.json`,
2. kopiuje go do `~/.kaggle/kaggle.json`,
3. nadaje mu uprawnienia `600`,
4. pobiera konkurs do lokalnego folderu `./data`,
5. rozpakowuje zipy,
6. opcjonalnie klonuje repo projektu do `./PigPostureComputerVision`.

Jeśli dane są już w `./data`, pobieranie jest pomijane.


In [ ]:
# ============================================================
# Lokalna konfiguracja Kaggle API
# ============================================================

kaggle_home = Path.home() / ".kaggle"
kaggle_target = kaggle_home / "kaggle.json"

if not kaggle_target.exists():
    if not CFG.KAGGLE_JSON.exists():
        raise FileNotFoundError(
            f"Nie znaleziono {CFG.KAGGLE_JSON}.
"
            "Pobierz kaggle.json z Kaggle Account -> API -> Create New Token "
            "i połóż plik w tym samym folderze co notebook."
        )
    kaggle_home.mkdir(parents=True, exist_ok=True)
    shutil.copy2(CFG.KAGGLE_JSON, kaggle_target)
    try:
        os.chmod(kaggle_target, 0o600)
    except Exception as e:
        print("[WARN] Nie udało się ustawić chmod 600, kontynuuję:", e)
    print("Skopiowano kaggle.json do:", kaggle_target)
else:
    print("Kaggle credentials już istnieją:", kaggle_target)

# Kaggle CLI czasem korzysta z tej zmiennej środowiskowej.
os.environ["KAGGLE_CONFIG_DIR"] = str(kaggle_home)

# ============================================================
# Pobranie danych konkursowych
# ============================================================

CFG.DATA_DIR.mkdir(parents=True, exist_ok=True)

if not any(CFG.DATA_DIR.glob("*")):
    print("DATA_DIR jest pusty. Pobieram dane konkursowe z Kaggle...")
    cmd = [
        sys.executable,
        "-m",
        "kaggle",
        "competitions",
        "download",
        "-c",
        CFG.COMPETITION,
        "-p",
        str(CFG.DATA_DIR),
        "--force",
    ]
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)

    for z in CFG.DATA_DIR.glob("*.zip"):
        print("Unzip:", z)
        with zipfile.ZipFile(z, "r") as zip_ref:
            zip_ref.extractall(CFG.DATA_DIR)
else:
    print("DATA_DIR nie jest pusty, pomijam pobieranie:", CFG.DATA_DIR)

# ============================================================
# Opcjonalne sklonowanie repo projektu
# ============================================================

if not CFG.REPO_DIR.exists():
    print("Klonuję repo projektu...")
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/StaryDron/PigPostureComputerVision.git", str(CFG.REPO_DIR)],
        check=False,
    )
else:
    print("Repo już istnieje:", CFG.REPO_DIR)

print("
Pliki w DATA_DIR:")
for p in sorted(CFG.DATA_DIR.glob("*"))[:80]:
    print(" -", p)


## 3. Automatyczne wykrycie plików CSV, obrazów i schematu danych

Ponieważ konkursowe paczki bywają różnie rozpakowane, poniższe funkcje:

- szukają `train.csv`, `test.csv`, `sample_submission.csv`,
- wykrywają kolumnę z ID obrazu,
- wykrywają kolumnę z etykietą,
- wykrywają bbox w formacie `x,y,w,h` albo `xmin,ymin,xmax,ymax`,
- budują indeks ścieżek obrazów po nazwie i stemie.

In [ ]:
def list_csv_files(root: Path):
    return sorted(root.rglob("*.csv"))

def read_csv_candidates(root: Path):
    rows = []
    for p in list_csv_files(root):
        try:
            df = pd.read_csv(p)
            rows.append({
                "path": str(p),
                "rows": len(df),
                "cols": list(df.columns),
                "head": df.head(2).to_dict(orient="records")
            })
        except Exception as e:
            rows.append({"path": str(p), "error": str(e)})
    return rows

csv_info = read_csv_candidates(CFG.DATA_DIR)
for info in csv_info:
    print("\nCSV:", info["path"])
    print("Rows:", info.get("rows"))
    print("Cols:", info.get("cols"))
    print("Head:", info.get("head"))

def choose_csvs(csv_info):
    train_csv = None
    test_csv = None
    sub_csv = None

    for info in csv_info:
        p = Path(info["path"])
        name = p.name.lower()
        cols = [c.lower() for c in info.get("cols", [])]

        if "sample" in name and "submission" in name:
            sub_csv = p
        elif "train" in name and any(c in cols for c in ["label", "class", "class_id", "target", "posture"]):
            train_csv = p
        elif "test" in name and not any(c in cols for c in ["label", "class", "class_id", "target", "posture"]):
            test_csv = p

    # Fallback: heurystyka po obecności labela.
    if train_csv is None:
        for info in csv_info:
            cols = [c.lower() for c in info.get("cols", [])]
            if any(c in cols for c in ["label", "class", "class_id", "target", "posture"]):
                train_csv = Path(info["path"])
                break

    if sub_csv is None:
        for info in csv_info:
            if "submission" in Path(info["path"]).name.lower():
                sub_csv = Path(info["path"])
                break

    if test_csv is None:
        for info in csv_info:
            p = Path(info["path"])
            if p != train_csv and p != sub_csv:
                test_csv = p
                break

    return train_csv, test_csv, sub_csv

TRAIN_CSV, TEST_CSV, SUB_CSV = choose_csvs(csv_info)
print("\nSelected:")
print("TRAIN_CSV:", TRAIN_CSV)
print("TEST_CSV :", TEST_CSV)
print("SUB_CSV  :", SUB_CSV)

train_df_raw = pd.read_csv(TRAIN_CSV)
test_df_raw = pd.read_csv(TEST_CSV) if TEST_CSV else None
sub_df = pd.read_csv(SUB_CSV) if SUB_CSV else None

display(train_df_raw.head())
if test_df_raw is not None:
    display(test_df_raw.head())
if sub_df is not None:
    display(sub_df.head())

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def build_image_index(root: Path):
    index = {}
    all_images = []
    for p in root.rglob("*"):
        if p.suffix.lower() in IMG_EXTS:
            all_images.append(p)
            # indeks po pełnej nazwie oraz stemie
            index[p.name] = p
            index[p.stem] = p
    return index, all_images

image_index, all_images = build_image_index(CFG.DATA_DIR)
print("Liczba wykrytych obrazów:", len(all_images))
print("Przykłady:")
for p in all_images[:10]:
    print(" -", p)

def infer_id_col(df):
    candidates = ["image_id", "id", "filename", "file_name", "image", "img", "path"]
    lower = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c in lower:
            return lower[c]
    # fallback: pierwsza kolumna tekstowa z wartościami wyglądającymi jak nazwy plików
    for c in df.columns:
        if df[c].dtype == "object":
            return c
    return df.columns[0]

def infer_label_col(df):
    candidates = ["label", "class", "class_id", "target", "posture", "category"]
    lower = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c in lower:
            return lower[c]
    # fallback: kolumna numeryczna z małą liczbą unikalnych wartości
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]) and df[c].nunique() <= 20:
            return c
    raise ValueError("Nie udało się wykryć kolumny etykiety.")

def infer_bbox_cols(df):
    lower = {c.lower(): c for c in df.columns}

    xywh_sets = [
        ("x", "y", "width", "height"),
        ("x", "y", "w", "h"),
        ("bbox_x", "bbox_y", "bbox_width", "bbox_height"),
    ]
    for names in xywh_sets:
        if all(n in lower for n in names):
            return tuple(lower[n] for n in names), "xywh"

    xyxy_sets = [
        ("xmin", "ymin", "xmax", "ymax"),
        ("x_min", "y_min", "x_max", "y_max"),
        ("left", "top", "right", "bottom"),
    ]
    for names in xyxy_sets:
        if all(n in lower for n in names):
            return tuple(lower[n] for n in names), "xyxy"

    # Kolumna bbox jako string/lista, np. "[x, y, w, h]"
    for c in df.columns:
        if "bbox" in c.lower():
            return (c,), "bbox_string"

    return None, None

ID_COL = infer_id_col(train_df_raw)
LABEL_COL = infer_label_col(train_df_raw)
BBOX_COLS, BBOX_MODE = infer_bbox_cols(train_df_raw)

print("ID_COL:", ID_COL)
print("LABEL_COL:", LABEL_COL)
print("BBOX_COLS:", BBOX_COLS, "BBOX_MODE:", BBOX_MODE)

## 4. Standaryzacja dataframe'ów

Tworzymy wspólny format:

```text
image_id | image_path | label | bbox_x1 | bbox_y1 | bbox_x2 | bbox_y2 | camera_group | event_group
```

`camera_group` służy do walidacji odpornej na shift między kamerami.  
`event_group` można użyć do uśrednienia predykcji między widokami, jeśli nazwa pliku zawiera informację o tym samym momencie z różnych kamer.

In [ ]:
def find_image_path(value, image_index):
    s = str(value)
    candidates = [
        s,
        Path(s).name,
        Path(s).stem,
    ]
    for c in candidates:
        if c in image_index:
            return str(image_index[c])
    # fallback: jeśli CSV zawiera relatywną ścieżkę
    p = CFG.DATA_DIR / s
    if p.exists():
        return str(p)
    return None

def parse_bbox(row, bbox_cols, bbox_mode):
    if bbox_cols is None:
        return [np.nan, np.nan, np.nan, np.nan]

    if bbox_mode == "xywh":
        x, y, w, h = [float(row[c]) for c in bbox_cols]
        return [x, y, x + w, y + h]

    if bbox_mode == "xyxy":
        x1, y1, x2, y2 = [float(row[c]) for c in bbox_cols]
        return [x1, y1, x2, y2]

    if bbox_mode == "bbox_string":
        raw = row[bbox_cols[0]]
        if pd.isna(raw):
            return [np.nan, np.nan, np.nan, np.nan]
        # Obsługa stringów typu "[x, y, w, h]" albo "x y w h"
        nums = re.findall(r"-?\d+\.?\d*", str(raw))
        nums = [float(x) for x in nums]
        if len(nums) >= 4:
            x, y, w, h = nums[:4]
            # Zakładamy xywh, bo to najczęstszy format bbox w CSV.
            # Jeśli w danych okaże się, że to xyxy, zmień poniższą linię.
            return [x, y, x + w, y + h]

    return [np.nan, np.nan, np.nan, np.nan]

def parse_camera_group(image_id):
    s = str(image_id)
    # W repo używaliście wzorca typu: pen2_orb_cam1_...
    m = re.search(r"(pen\d+)_(orb|tur)_(cam\d+)", s)
    if m:
        return "_".join(m.groups())
    # fallback: jeśli w nazwie są podobne tokeny
    m = re.search(r"(cam\d+)", s)
    if m:
        return m.group(1)
    return "unknown_camera"

def parse_event_group(image_id):
    s = str(image_id)
    # Usuwamy segment kamery; jeśli ten sam moment występuje w różnych kamerach,
    # powinien dostać podobny klucz.
    s2 = re.sub(r"(pen\d+)_(orb|tur)_(cam\d+)_?", "", s)
    s2 = re.sub(r"(cam\d+)_?", "", s2)
    s2 = Path(s2).stem
    return s2 if s2 else Path(str(image_id)).stem

def make_standard_df(df, is_train=True):
    out = pd.DataFrame()
    out["image_id"] = df[ID_COL].astype(str)
    out["image_path"] = out["image_id"].apply(lambda x: find_image_path(x, image_index))

    if is_train:
        raw_labels = df[LABEL_COL]
        # Jeśli etykiety są tekstowe, mapujemy po CFG.CLASS_NAMES albo alfabetycznie.
        if raw_labels.dtype == "object":
            name_to_id = {name: i for i, name in enumerate(CFG.CLASS_NAMES)}
            if set(raw_labels.unique()).issubset(set(name_to_id.keys())):
                out["label"] = raw_labels.map(name_to_id).astype(int)
            else:
                classes = sorted(raw_labels.unique())
                print("Wykryte tekstowe klasy:", classes)
                name_to_id = {name: i for i, name in enumerate(classes)}
                out["label"] = raw_labels.map(name_to_id).astype(int)
                CFG.CLASS_NAMES = classes
                CFG.NUM_CLASSES = len(classes)
        else:
            out["label"] = raw_labels.astype(int)

    bboxes = df.apply(lambda r: parse_bbox(r, BBOX_COLS, BBOX_MODE), axis=1)
    bboxes = pd.DataFrame(bboxes.tolist(), columns=["bbox_x1", "bbox_y1", "bbox_x2", "bbox_y2"])
    out = pd.concat([out, bboxes], axis=1)

    out["camera_group"] = out["image_id"].apply(parse_camera_group)
    out["event_group"] = out["image_id"].apply(parse_event_group)

    # Dodatkowe grupowanie po bazowej nazwie, aby uniknąć przecieku podobnych klatek.
    out["image_group"] = out["image_id"].apply(lambda x: Path(str(x)).stem.split("_bbox")[0])

    return out

train_df = make_standard_df(train_df_raw, is_train=True)
test_df = make_standard_df(test_df_raw, is_train=False) if test_df_raw is not None else None

missing_paths = train_df["image_path"].isna().sum()
print("Brakujące ścieżki train:", missing_paths, "/", len(train_df))
if test_df is not None:
    print("Brakujące ścieżki test:", test_df["image_path"].isna().sum(), "/", len(test_df))

display(train_df.head())
print("\nRozkład klas:")
display(train_df["label"].value_counts().sort_index().rename(index=dict(enumerate(CFG.CLASS_NAMES))))
print("\nRozkład kamer:")
display(train_df["camera_group"].value_counts().head(20))

## 5. Tworzenie foldów bez przecieku domenowego

To jest jedna z najważniejszych zmian względem prostego losowego splita.

Domyślnie próbujemy `GroupKFold` po `camera_group`. Jeśli kamer jest za mało, fallback do `image_group`, a potem do `StratifiedKFold`.

Celem jest wykrycie sytuacji: model dobrze zna tło/kamerę, ale słabo generalizuje na nowy widok.

In [ ]:
def add_folds(df):
    df = df.copy()
    df["fold"] = -1
    y = df["label"].values

    if CFG.CV_MODE == "camera_group":
        groups = df["camera_group"].values
    elif CFG.CV_MODE == "image_group":
        groups = df["image_group"].values
    else:
        groups = None

    n_unique_groups = len(pd.unique(groups)) if groups is not None else 0

    if groups is not None and n_unique_groups >= CFG.MIN_GROUPS_FOR_GROUPKOLD:
        n_splits = min(CFG.N_FOLDS, n_unique_groups)
        splitter = GroupKFold(n_splits=n_splits)
        print(f"Używam GroupKFold: mode={CFG.CV_MODE}, groups={n_unique_groups}, folds={n_splits}")
        for fold, (_, val_idx) in enumerate(splitter.split(df, y, groups)):
            df.loc[val_idx, "fold"] = fold
    else:
        n_splits = min(CFG.N_FOLDS, df["label"].value_counts().min())
        n_splits = max(2, n_splits)
        splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=CFG.SEED)
        print(f"Fallback do StratifiedKFold, folds={n_splits}")
        for fold, (_, val_idx) in enumerate(splitter.split(df, y)):
            df.loc[val_idx, "fold"] = fold

    return df

train_df = add_folds(train_df)

fold_summary = train_df.groupby(["fold", "label"]).size().unstack(fill_value=0)
display(fold_summary)

if "camera_group" in train_df.columns:
    print("Kamery per fold:")
    display(train_df.groupby("fold")["camera_group"].nunique())

## 6. Dataset: bbox crop, padding i label-aware horizontal flip

Najważniejsze poprawki:

- **nie używamy `VerticalFlip`**,
- **nie używamy rotacji 90°**,
- poziomy flip robi osobna logika w `Dataset`, bo trzeba zmienić etykietę `0 ↔ 1`,
- transformacje geometryczne są umiarkowane,
- crop z bboxem dostaje padding, aby zachować kontekst.

In [ ]:
def crop_with_padding(img, bbox, padding=0.2):
    h, w = img.shape[:2]
    x1, y1, x2, y2 = bbox

    if any(pd.isna(v) for v in [x1, y1, x2, y2]):
        return img

    x1, y1, x2, y2 = map(float, [x1, y1, x2, y2])
    bw = max(1.0, x2 - x1)
    bh = max(1.0, y2 - y1)

    pad_x = bw * padding
    pad_y = bh * padding

    nx1 = int(max(0, x1 - pad_x))
    ny1 = int(max(0, y1 - pad_y))
    nx2 = int(min(w, x2 + pad_x))
    ny2 = int(min(h, y2 + pad_y))

    if nx2 <= nx1 or ny2 <= ny1:
        return img

    return img[ny1:ny2, nx1:nx2]

def get_train_transforms(img_size):
    return A.Compose([
        A.LongestMaxSize(max_size=img_size),
        A.PadIfNeeded(min_height=img_size, min_width=img_size, border_mode=cv2.BORDER_CONSTANT, value=0),
        A.RandomResizedCrop(size=(img_size, img_size), scale=(0.80, 1.00), ratio=(0.85, 1.15), p=0.7),

        # Umiarkowane augmentacje fotometryczne — realistyczne dla różnych kamer i oświetlenia.
        A.OneOf([
            A.RandomBrightnessContrast(brightness_limit=0.18, contrast_limit=0.18),
            A.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.03),
            A.CLAHE(clip_limit=2.0),
        ], p=0.6),

        # Małe rotacje i przesunięcia. Bez 90° i bez vertical flip.
        A.ShiftScaleRotate(
            shift_limit=0.04,
            scale_limit=0.08,
            rotate_limit=12,
            border_mode=cv2.BORDER_CONSTANT,
            value=0,
            p=0.45,
        ),

        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 5)),
            A.MotionBlur(blur_limit=3),
            A.GaussNoise(var_limit=(5.0, 25.0)),
        ], p=0.20),

        A.CoarseDropout(
            max_holes=6,
            max_height=int(img_size * 0.08),
            max_width=int(img_size * 0.08),
            min_holes=1,
            fill_value=0,
            p=0.20,
        ),

        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

def get_valid_transforms(img_size):
    return A.Compose([
        A.LongestMaxSize(max_size=img_size),
        A.PadIfNeeded(min_height=img_size, min_width=img_size, border_mode=cv2.BORDER_CONSTANT, value=0),
        A.CenterCrop(height=img_size, width=img_size),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

class PigPostureDataset(Dataset):
    def __init__(
        self,
        df,
        transforms=None,
        is_train=True,
        camera_to_idx=None,
        hflip_prob=0.5,
        pseudo_weight_col=None,
    ):
        self.df = df.reset_index(drop=True)
        self.transforms = transforms
        self.is_train = is_train
        self.camera_to_idx = camera_to_idx or {"unknown_camera": 0}
        self.hflip_prob = hflip_prob if is_train else 0.0
        self.pseudo_weight_col = pseudo_weight_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row["image_path"]

        img = cv2.imread(str(img_path))
        if img is None:
            raise FileNotFoundError(f"Nie udało się odczytać obrazu: {img_path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        bbox = [row["bbox_x1"], row["bbox_y1"], row["bbox_x2"], row["bbox_y2"]]
        img = crop_with_padding(img, bbox, padding=CFG.CROP_PADDING)

        target = int(row["label"]) if "label" in row else -1

        # Label-aware horizontal flip.
        # To naprawia kluczowy błąd: left/right muszą zostać zamienione po odbiciu.
        if self.is_train and random.random() < self.hflip_prob:
            img = cv2.flip(img, 1)
            if target >= 0:
                target = CFG.FLIP_LABEL_MAP.get(target, target)

        if self.transforms:
            img = self.transforms(image=img)["image"]

        camera = str(row.get("camera_group", "unknown_camera"))
        camera_idx = self.camera_to_idx.get(camera, self.camera_to_idx.get("unknown_camera", 0))

        item = {
            "image": img,
            "camera_idx": torch.tensor(camera_idx, dtype=torch.long),
            "image_id": str(row["image_id"]),
        }

        if target >= 0:
            item["label"] = torch.tensor(target, dtype=torch.long)

        if self.pseudo_weight_col and self.pseudo_weight_col in row:
            item["sample_weight"] = torch.tensor(float(row[self.pseudo_weight_col]), dtype=torch.float32)
        else:
            item["sample_weight"] = torch.tensor(1.0, dtype=torch.float32)

        return item

## 7. Model: backbone + opcjonalny embedding kamery

Zamiast traktować każdy obraz całkowicie niezależnie od kamery, dokładamy mały embedding `camera_group`.  
To pozwala modelowi nauczyć się różnic między widokami, bez ręcznego kodowania reguł.

In [ ]:
class PigPostureModel(nn.Module):
    def __init__(self, model_name, num_classes, num_cameras=1, use_camera_embedding=True, camera_emb_dim=16, pretrained=True):
        super().__init__()
        self.use_camera_embedding = use_camera_embedding

        # num_classes=0 oznacza, że timm zwraca embedding zamiast gotowych logitów.
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0, global_pool="avg")
        feat_dim = self.backbone.num_features

        if use_camera_embedding:
            self.camera_emb = nn.Embedding(num_cameras, camera_emb_dim)
            head_in = feat_dim + camera_emb_dim
        else:
            self.camera_emb = None
            head_in = feat_dim

        self.head = nn.Sequential(
            nn.LayerNorm(head_in),
            nn.Dropout(0.25),
            nn.Linear(head_in, num_classes),
        )

    def forward(self, x, camera_idx=None):
        feat = self.backbone(x)

        if self.use_camera_embedding:
            if camera_idx is None:
                camera_idx = torch.zeros(x.size(0), dtype=torch.long, device=x.device)
            cam_feat = self.camera_emb(camera_idx)
            feat = torch.cat([feat, cam_feat], dim=1)

        return self.head(feat)

## 8. Loss, scheduler i metryki

Optymalizujemy pod **macro F1**, więc monitorujemy F1 per klasa, nie tylko loss.  
Loss domyślnie to `CrossEntropyLoss` z wagami klas i lekkim label smoothingiem.

In [ ]:
def compute_class_weights(labels, num_classes):
    counts = np.bincount(labels, minlength=num_classes)
    counts = np.maximum(counts, 1)
    weights = counts.sum() / (num_classes * counts)
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32)

def make_weighted_sampler(labels):
    counts = np.bincount(labels, minlength=CFG.NUM_CLASSES)
    counts = np.maximum(counts, 1)
    sample_weights = 1.0 / counts[labels]
    return WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

def get_lr(optimizer):
    return optimizer.param_groups[0]["lr"]

def make_scheduler(optimizer, num_train_steps):
    # Cosine z warmupem liczonym po stepach.
    warmup_steps = max(1, int(CFG.WARMUP_EPOCHS * num_train_steps / CFG.EPOCHS))

    def lr_lambda(step):
        if step < warmup_steps:
            return float(step + 1) / float(warmup_steps)
        progress = float(step - warmup_steps) / float(max(1, num_train_steps - warmup_steps))
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

def macro_f1_from_logits(logits, labels):
    preds = logits.argmax(axis=1)
    return f1_score(labels, preds, average="macro")

def print_fold_report(y_true, y_pred):
    print("Macro F1:", f1_score(y_true, y_pred, average="macro"))
    print("\nClassification report:")
    print(classification_report(y_true, y_pred, target_names=CFG.CLASS_NAMES, digits=4))
    print("\nConfusion matrix:")
    cm = confusion_matrix(y_true, y_pred, labels=list(range(CFG.NUM_CLASSES)))
    display(pd.DataFrame(cm, index=CFG.CLASS_NAMES, columns=CFG.CLASS_NAMES))

## 9. Pętla treningowa i walidacyjna

Ważne elementy:

- AMP dla szybszego treningu,
- gradient clipping,
- zapisywanie najlepszego checkpointu po macro F1,
- OOF predykcje do późniejszego ensemblingu i analizy błędów.

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler, criterion, scaler):
    model.train()
    running_loss = 0.0
    all_logits = []
    all_labels = []

    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)
        camera_idx = batch["camera_idx"].to(device, non_blocking=True)
        sample_weights = batch["sample_weight"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=CFG.USE_AMP):
            logits = model(images, camera_idx)
            loss_per_sample = F.cross_entropy(
                logits,
                labels,
                weight=criterion.weight,
                label_smoothing=CFG.LABEL_SMOOTHING,
                reduction="none",
            )
            loss = (loss_per_sample * sample_weights).mean()

        scaler.scale(loss).backward()

        if CFG.GRAD_CLIP is not None:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP)

        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running_loss += loss.item() * images.size(0)
        all_logits.append(logits.detach().cpu().float().numpy())
        all_labels.append(labels.detach().cpu().numpy())

    all_logits = np.concatenate(all_logits)
    all_labels = np.concatenate(all_labels)
    return running_loss / len(loader.dataset), macro_f1_from_logits(all_logits, all_labels)

@torch.no_grad()
def valid_one_epoch(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    all_logits = []
    all_labels = []
    all_ids = []
    all_cameras = []

    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)
        camera_idx = batch["camera_idx"].to(device, non_blocking=True)

        logits = model(images, camera_idx)
        loss = criterion(logits, labels)

        running_loss += loss.item() * images.size(0)
        all_logits.append(logits.detach().cpu().float().numpy())
        all_labels.append(labels.detach().cpu().numpy())
        all_ids.extend(batch["image_id"])
        all_cameras.extend(camera_idx.detach().cpu().numpy().tolist())

    all_logits = np.concatenate(all_logits)
    all_labels = np.concatenate(all_labels)
    return {
        "loss": running_loss / len(loader.dataset),
        "f1": macro_f1_from_logits(all_logits, all_labels),
        "logits": all_logits,
        "labels": all_labels,
        "image_ids": all_ids,
        "camera_indices": all_cameras,
    }

## 10. Full-auto: trening wszystkich foldów + finalny submission

Ta komórka zastępuje wcześniejszy tryb ręcznego odpalania pojedynczego folda.

Co robi automatycznie:

- trenuje wszystkie foldy znalezione w `train_df["fold"]`,
- zapisuje najlepszy checkpoint każdego folda,
- zapisuje `history_fold*.csv`, `oof_fold*.csv`, `oof_all_folds.csv`, `history_all_folds.csv`,
- naprawia problem z `torch.load` w PyTorch 2.6+ przez bezpieczną serializację konfiguracji i jawne `weights_only=False`,
- po zakończeniu robi submission jako ensemble wszystkich checkpointów foldowych.

Przed uruchomieniem tej komórki skonfiguruj model w komórce `CFG`, np. `CFG.MODEL_NAME`, `CFG.IMG_SIZE`, `CFG.BATCH_SIZE`, `CFG.EPOCHS`, `CFG.LR`.


In [ ]:
# ============================================================
# FULL AUTO TRAINING: wszystkie foldy + bezpieczne checkpointy + submission
# ============================================================

from pathlib import Path
import json
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score

# ------------------------------------------------------------
# 1. Bezpieczna serializacja CFG
# ------------------------------------------------------------

def make_serializable_value(v):
    """
    Zamienia obiekty, które mogą psuć torch.load w PyTorch 2.6+,
    na proste typy: str, int, float, bool, list, dict, None.
    """
    if isinstance(v, Path):
        return str(v)

    if isinstance(v, (str, int, float, bool)) or v is None:
        return v

    if isinstance(v, (list, tuple)):
        return [make_serializable_value(x) for x in v]

    if isinstance(v, dict):
        return {str(k): make_serializable_value(val) for k, val in v.items()}

    # Fallback: zapisujemy reprezentację tekstową.
    return str(v)


def make_serializable_cfg(cfg):
    """
    Zapisujemy tylko atrybuty konfiguracyjne pisane dużymi literami.
    Dzięki temu checkpoint nie zawiera pathlib.Path ani innych obiektów,
    które powodowały błąd UnpicklingError.
    """
    serializable = {}

    for k, v in cfg.__dict__.items():
        if not k.isupper():
            continue
        serializable[k] = make_serializable_value(v)

    return serializable

# ------------------------------------------------------------
# 2. Upewniamy się, że katalog wyjściowy istnieje
# ------------------------------------------------------------

CFG.OUT_DIR = Path(CFG.OUT_DIR)
CFG.OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output dir:", CFG.OUT_DIR)

# ------------------------------------------------------------
# 3. Kontrola etykiet i mapowanie kamera -> indeks do embeddingu
# ------------------------------------------------------------

expected_labels = set(range(CFG.NUM_CLASSES))
actual_labels = set(pd.Series(train_df["label"]).dropna().astype(int).unique().tolist())
bad_labels = sorted(actual_labels - expected_labels)
if bad_labels:
    raise ValueError(
        f"train_df['label'] zawiera niepoprawne klasy: {bad_labels}. "
        f"Oczekiwano tylko {sorted(expected_labels)}. "
        "To zwykle oznacza, że wcześniej źle wykryto kolumnę label."
    )
train_df["label"] = train_df["label"].astype(int)

all_cameras = sorted(train_df["camera_group"].fillna("unknown_camera").unique().tolist())

if "unknown_camera" not in all_cameras:
    all_cameras = ["unknown_camera"] + all_cameras

camera_to_idx = {c: i for i, c in enumerate(all_cameras)}
idx_to_camera = {i: c for c, i in camera_to_idx.items()}

print("Liczba kamer:", len(camera_to_idx))
print(camera_to_idx)

# ------------------------------------------------------------
# 4. Funkcja treningu pojedynczego folda
# ------------------------------------------------------------

def run_fold(fold):
    seed_everything(CFG.SEED + int(fold))

    trn_df = train_df[train_df["fold"] != fold].reset_index(drop=True)
    val_df = train_df[train_df["fold"] == fold].reset_index(drop=True)

    if len(val_df) == 0:
        raise ValueError(f"Fold {fold} ma pusty valid set. Sprawdź train_df['fold'].")

    print("\n" + "=" * 100)
    print(f"FOLD {fold}")
    print("=" * 100)
    print(f"Train rows: {len(trn_df)}")
    print(f"Valid rows: {len(val_df)}")
    print("Valid cameras:", sorted(val_df["camera_group"].dropna().unique().tolist())[:30])

    print("\nTrain class distribution:")
    print(trn_df["label"].value_counts().sort_index())

    print("\nValid class distribution:")
    print(val_df["label"].value_counts().sort_index())

    train_ds = PigPostureDataset(
        trn_df,
        transforms=get_train_transforms(CFG.IMG_SIZE),
        is_train=True,
        camera_to_idx=camera_to_idx,
        hflip_prob=0.5,
    )

    valid_ds = PigPostureDataset(
        val_df,
        transforms=get_valid_transforms(CFG.IMG_SIZE),
        is_train=False,
        camera_to_idx=camera_to_idx,
    )

    sampler = None
    shuffle = True

    if CFG.USE_WEIGHTED_SAMPLER:
        sampler = make_weighted_sampler(trn_df["label"].values)
        shuffle = False

    train_loader = DataLoader(
        train_ds,
        batch_size=CFG.BATCH_SIZE,
        shuffle=shuffle,
        sampler=sampler,
        num_workers=CFG.NUM_WORKERS,
        pin_memory=True,
        drop_last=True,
    )

    valid_loader = DataLoader(
        valid_ds,
        batch_size=CFG.BATCH_SIZE * 2,
        shuffle=False,
        num_workers=CFG.NUM_WORKERS,
        pin_memory=True,
        drop_last=False,
    )

    model = PigPostureModel(
        model_name=CFG.MODEL_NAME,
        num_classes=CFG.NUM_CLASSES,
        num_cameras=len(camera_to_idx),
        use_camera_embedding=CFG.USE_CAMERA_EMBEDDING,
        camera_emb_dim=CFG.CAMERA_EMB_DIM,
        pretrained=True,
    ).to(device)

    class_weights = compute_class_weights(trn_df["label"].values, CFG.NUM_CLASSES).to(device)

    criterion = nn.CrossEntropyLoss(
        weight=class_weights,
        label_smoothing=CFG.LABEL_SMOOTHING,
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=CFG.LR,
        weight_decay=CFG.WEIGHT_DECAY,
    )

    num_train_steps = CFG.EPOCHS * len(train_loader)
    scheduler = make_scheduler(optimizer, num_train_steps)

    scaler = torch.cuda.amp.GradScaler(enabled=CFG.USE_AMP)

    best_f1 = -1.0

    safe_model_name = CFG.MODEL_NAME.replace("/", "_").replace(":", "_")
    best_path = CFG.OUT_DIR / f"{safe_model_name}_fold{fold}_best.pth"

    history = []

    for epoch in range(1, CFG.EPOCHS + 1):
        print("\n" + "-" * 80)
        print(f"Fold {fold} | Epoch {epoch}/{CFG.EPOCHS}")
        print("-" * 80)

        train_loss, train_f1 = train_one_epoch(
            model,
            train_loader,
            optimizer,
            scheduler,
            criterion,
            scaler,
        )

        val_result = valid_one_epoch(
            model,
            valid_loader,
            criterion,
        )

        row = {
            "fold": int(fold),
            "epoch": int(epoch),
            "lr": float(get_lr(optimizer)),
            "train_loss": float(train_loss),
            "train_f1": float(train_f1),
            "valid_loss": float(val_result["loss"]),
            "valid_f1": float(val_result["f1"]),
        }

        history.append(row)

        print(
            f"lr={row['lr']:.2e} | "
            f"train_loss={row['train_loss']:.4f} | "
            f"train_f1={row['train_f1']:.4f} | "
            f"valid_loss={row['valid_loss']:.4f} | "
            f"valid_f1={row['valid_f1']:.4f}"
        )

        if val_result["f1"] > best_f1:
            best_f1 = float(val_result["f1"])

            checkpoint = {
                "model": model.state_dict(),
                "cfg": make_serializable_cfg(CFG),
                "camera_to_idx": camera_to_idx,
                "idx_to_camera": idx_to_camera,
                "fold": int(fold),
                "best_f1": best_f1,
                "model_name": CFG.MODEL_NAME,
                "num_classes": CFG.NUM_CLASSES,
                "num_cameras": len(camera_to_idx),
                "use_camera_embedding": CFG.USE_CAMERA_EMBEDDING,
                "camera_emb_dim": CFG.CAMERA_EMB_DIM,
            }

            torch.save(checkpoint, best_path)

            print(f"  -> zapisano best checkpoint: {best_path}")
            print(f"  -> best_f1: {best_f1:.5f}")

    # --------------------------------------------------------
    # Wczytanie najlepszego checkpointu i finalne OOF dla folda
    # --------------------------------------------------------

    print("\nWczytuję najlepszy checkpoint folda:", best_path)

    ckpt = torch.load(
        best_path,
        map_location=device,
        weights_only=False,
    )

    model.load_state_dict(ckpt["model"])
    model.eval()

    val_result = valid_one_epoch(
        model,
        valid_loader,
        criterion,
    )

    oof_df = val_df.copy()

    for c in range(CFG.NUM_CLASSES):
        oof_df[f"logit_{c}"] = val_result["logits"][:, c]

    oof_df["pred"] = val_result["logits"].argmax(axis=1)

    print("\nFinal fold report:")
    print_fold_report(oof_df["label"].values, oof_df["pred"].values)

    hist_df = pd.DataFrame(history)

    hist_path = CFG.OUT_DIR / f"history_fold{fold}.csv"
    oof_path = CFG.OUT_DIR / f"oof_fold{fold}.csv"

    hist_df.to_csv(hist_path, index=False)
    oof_df.to_csv(oof_path, index=False)

    print("\nZapisano:")
    print("History:", hist_path)
    print("OOF:", oof_path)

    del model, train_loader, valid_loader, train_ds, valid_ds
    gc.collect()
    torch.cuda.empty_cache()

    return best_path, hist_df, oof_df

# ------------------------------------------------------------
# 5. Predykcja na test
# ------------------------------------------------------------

@torch.no_grad()
def predict_loader(model, loader):
    model.eval()

    all_logits = []
    all_ids = []

    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        camera_idx = batch["camera_idx"].to(device, non_blocking=True)

        logits = model(images, camera_idx)

        if CFG.USE_TTA and CFG.TTA_HFLIP:
            flipped = torch.flip(images, dims=[3])
            logits_flip = model(flipped, camera_idx)

            # Horizontal flip zamienia left/right, więc trzeba zamienić logity 0 i 1.
            logits_flip = logits_flip.clone()
            logits_flip[:, [0, 1]] = logits_flip[:, [1, 0]]

            logits = (logits + logits_flip) / 2.0

        all_logits.append(logits.detach().cpu().float().numpy())
        all_ids.extend(batch["image_id"])

    return np.concatenate(all_logits), all_ids


def load_model_from_checkpoint(path):
    """
    Ładowanie checkpointu z poprawką na PyTorch 2.6+.
    weights_only=False jest potrzebne, bo checkpoint zawiera słownik z metadanymi.
    """
    ckpt = torch.load(
        path,
        map_location=device,
        weights_only=False,
    )

    model = PigPostureModel(
        model_name=ckpt.get("model_name", CFG.MODEL_NAME),
        num_classes=ckpt.get("num_classes", CFG.NUM_CLASSES),
        num_cameras=ckpt.get("num_cameras", len(camera_to_idx)),
        use_camera_embedding=ckpt.get("use_camera_embedding", CFG.USE_CAMERA_EMBEDDING),
        camera_emb_dim=ckpt.get("camera_emb_dim", CFG.CAMERA_EMB_DIM),
        pretrained=False,
    ).to(device)

    model.load_state_dict(ckpt["model"])
    model.eval()

    return model, ckpt


def aggregate_by_event_group(test_df, logits):
    """
    Opcjonalna agregacja po event_group.
    Jeśli event_group jest dobrze zdefiniowane, może stabilizować predykcje.
    Jeśli nie jesteś pewien, ustaw CFG.USE_GROUP_AGGREGATION = False.
    """
    out_logits = logits.copy()

    tmp = test_df[["event_group"]].copy()

    for c in range(logits.shape[1]):
        tmp[f"logit_{c}"] = logits[:, c]

    logit_cols = [f"logit_{c}" for c in range(logits.shape[1])]
    agg = tmp.groupby("event_group")[logit_cols].mean()

    for i, eg in enumerate(test_df["event_group"].values):
        out_logits[i] = agg.loc[eg].values

    return out_logits


def make_submission(checkpoint_paths, submission_path=None):
    assert test_df is not None, "Nie znaleziono test_df."

    print("\n" + "=" * 100)
    print("INFERENCE / SUBMISSION")
    print("=" * 100)

    test_ds = PigPostureDataset(
        test_df,
        transforms=get_valid_transforms(CFG.IMG_SIZE),
        is_train=False,
        camera_to_idx=camera_to_idx,
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=CFG.BATCH_SIZE * 2,
        shuffle=False,
        num_workers=CFG.NUM_WORKERS,
        pin_memory=True,
        drop_last=False,
    )

    logits_ensemble = None
    ids = None

    for path in checkpoint_paths:
        print("Predict:", path)

        model, ckpt = load_model_from_checkpoint(path)
        logits, ids = predict_loader(model, test_loader)

        if logits_ensemble is None:
            logits_ensemble = logits
        else:
            logits_ensemble += logits

        del model
        gc.collect()
        torch.cuda.empty_cache()

    logits_ensemble = logits_ensemble / len(checkpoint_paths)

    if CFG.USE_GROUP_AGGREGATION:
        print("Agreguję predykcje po event_group.")
        logits_ensemble = aggregate_by_event_group(test_df, logits_ensemble)

    preds = logits_ensemble.argmax(axis=1)

    # --------------------------------------------------------
    # Budowa submission
    # --------------------------------------------------------

    if sub_df is not None:
        submission = sub_df.copy()
        sub_id_col = infer_id_col(submission)

        pred_candidates = ["label", "class", "class_id", "target", "posture"]
        pred_col = None

        lower_to_original = {x.lower(): x for x in submission.columns}

        for c in pred_candidates:
            if c in lower_to_original:
                candidate = lower_to_original[c]
                if candidate != sub_id_col:
                    pred_col = candidate
                    break

        if pred_col is None or pred_col == sub_id_col:
            non_id = [c for c in submission.columns if c != sub_id_col]
            pred_col = non_id[0] if len(non_id) > 0 else "label"

        pred_map = dict(zip(test_df["image_id"].astype(str), preds))
        mapped = submission[sub_id_col].astype(str).map(pred_map)

        # Jeśli sample_submission ma inną kolejność lub inne ID, awaryjnie wypełniamy po kolejności.
        fallback = pd.Series(preds[:len(submission)], index=submission.index)
        submission[pred_col] = mapped.fillna(fallback).astype(int)

    else:
        submission = pd.DataFrame(
            {
                "image_id": ids,
                "label": preds,
            }
        )

    if submission_path is None:
        submission_path = CFG.OUT_DIR / "submission.csv"

    submission.to_csv(submission_path, index=False)

    # Zapisujemy też logity ensemble do diagnostyki.
    logits_df = pd.DataFrame(
        logits_ensemble,
        columns=[f"logit_{c}" for c in range(CFG.NUM_CLASSES)],
    )
    logits_df.insert(0, "image_id", test_df["image_id"].values)
    logits_path = CFG.OUT_DIR / "test_logits_ensemble.csv"
    logits_df.to_csv(logits_path, index=False)

    print("\nSaved submission:", submission_path)
    print("Saved test logits:", logits_path)

    print("\nSubmission preview:")
    display(submission.head())

    print("\nPredicted class distribution:")
    print(pd.Series(preds).value_counts().sort_index())

    return submission, logits_ensemble

# ------------------------------------------------------------
# 6. Automatyczny trening wszystkich foldów
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("START FULL CROSS-VALIDATION TRAINING")
print("=" * 100)

folds_to_train = sorted([int(f) for f in train_df["fold"].dropna().unique().tolist()])

print("Foldy znalezione w train_df:", folds_to_train)

if len(folds_to_train) == 0:
    raise ValueError("Nie znaleziono żadnych foldów w train_df['fold'].")

# Jeśli chcesz koniecznie dokładnie 5 foldów, odkomentuj:
# assert folds_to_train == [0, 1, 2, 3, 4], f"Oczekiwano foldów [0,1,2,3,4], a znaleziono: {folds_to_train}"

all_oof = []
all_checkpoints = []
all_histories = []

for fold in folds_to_train:
    best_path, hist_df, fold_oof = run_fold(fold)

    all_checkpoints.append(best_path)
    all_oof.append(fold_oof)

    hist_df = hist_df.copy()
    hist_df["fold"] = fold
    all_histories.append(hist_df)

    gc.collect()
    torch.cuda.empty_cache()

all_oof = pd.concat(all_oof, ignore_index=True)
all_histories = pd.concat(all_histories, ignore_index=True)

oof_path = CFG.OUT_DIR / "oof_all_folds.csv"
hist_path = CFG.OUT_DIR / "history_all_folds.csv"
checkpoints_path = CFG.OUT_DIR / "checkpoints.json"

all_oof.to_csv(oof_path, index=False)
all_histories.to_csv(hist_path, index=False)

with open(checkpoints_path, "w") as f:
    json.dump([str(p) for p in all_checkpoints], f, indent=2)

global_oof_f1 = f1_score(
    all_oof["label"].values,
    all_oof["pred"].values,
    average="macro",
)

print("\n" + "=" * 100)
print("FULL CV FINISHED")
print("=" * 100)

print("Global OOF F1:", global_oof_f1)

print("\nZapisano:")
print("OOF:", oof_path)
print("History:", hist_path)
print("Checkpoints list:", checkpoints_path)

print("\nCheckpointy:")
for p in all_checkpoints:
    print(p)

print("\nOOF preview:")
display(all_oof.head())

print("\nHistory preview:")
display(all_histories.head())

# ------------------------------------------------------------
# 7. Finalny submission jako ensemble wszystkich foldów
# ------------------------------------------------------------

submission_path = CFG.OUT_DIR / "submission_ensemble_all_folds.csv"

submission, test_logits = make_submission(
    all_checkpoints,
    submission_path=submission_path,
)

print("\n" + "=" * 100)
print("GOTOWE")
print("=" * 100)
print("Final submission:", submission_path)


## 11. Lokalizacja wyników

Wersja lokalna zapisuje wszystko w folderze:

```text
./pig_posture_work/outputs/
```

Najważniejszy plik po zakończeniu treningu:

```text
./pig_posture_work/outputs/submission_ensemble_all_folds.csv
```


In [ ]:
# Lista najważniejszych wyników zapisanych lokalnie.

print("OUT_DIR:", CFG.OUT_DIR.resolve())

if CFG.OUT_DIR.exists():
    files = sorted(CFG.OUT_DIR.glob("*"), key=lambda p: p.stat().st_mtime, reverse=True)
    for p in files[:80]:
        size_mb = p.stat().st_size / (1024 ** 2)
        print(f"{size_mb:8.2f} MB | {p.name}")
else:
    print("OUT_DIR jeszcze nie istnieje.")
